# Step 5: 커맨드 시스템 — 슬래시 명령어로 워크플로우 제어

## 학습 목표

- Claude Code의 **3가지 커맨드 타입**(PromptCommand, LocalCommand, LocalJSXCommand)의 설계를 이해합니다
- **PromptCommand**가 잘 작성된 프롬프트 하나로 복잡한 워크플로우를 구현하는 패턴을 배웁니다
- **CommandRegistry**로 커맨드 파싱, 별칭(alias), Feature Flag 조건부 등록을 구현합니다

> **커맨드 시스템이란?**
> 사용자가 `/commit`, `/review`, `/cost` 같은 슬래시 명령어를 입력하면,
> 에이전트 루프를 거치지 않고 직접 실행되거나, 최적화된 프롬프트로 변환되어 LLM에 전달됩니다.
> 50개 이상의 커맨드가 이 시스템으로 동작합니다.

---

## Claude Code 소스 분석

### 5-1. 커맨드 시스템의 전체 그림

Claude Code의 커맨드는 세 가지 타입으로 나뉘며, 각각 다른 실행 경로를 탑니다:

```
사용자 입력: "/commit fix login bug"
       │
       ▼
CommandRegistry.parse("/commit fix login bug")
       │
       ├─ PromptCommand  ──→  getPromptForCommand(args) ─→ LLM API ─→ 응답
       │   예: /commit, /review, /explain
       │   → 프롬프트를 생성하여 에이전트 루프에 전달
       │
       ├─ LocalCommand   ──→  call(args, context) ─→ 텍스트 반환
       │   예: /cost, /version, /status
       │   → LLM 없이 로컬에서 즉시 실행
       │
       └─ LocalJSXCommand ─→  call(args, context) ─→ React JSX 반환
           예: /help, /doctor, /config
           → 터미널 UI 컴포넌트를 렌더링
```

**핵심 통찰:** `/commit`은 코드를 한 줄도 작성하지 않습니다. 대신 "git diff를 분석하고 커밋 메시지를 작성하라"는 프롬프트를 LLM에 보냅니다. **프롬프트가 코드를 대체하는** 패턴입니다.

### 5-2. 3가지 커맨드 타입 정의 (commands.ts)

```typescript
// src/commands.ts — 커맨드 타입 정의

interface PromptCommand {
  type: 'prompt'
  name: string
  description: string
  // 사용자 인자를 받아 LLM에 보낼 프롬프트를 생성
  getPromptForCommand(args: string): string
}

interface LocalCommand {
  type: 'local'
  name: string
  description: string
  // LLM 없이 로컬에서 실행, 텍스트 반환
  call(args: string, context: ToolUseContext): string
}

interface LocalJSXCommand {
  type: 'local-jsx'
  name: string
  description: string
  // 로컬에서 실행, React JSX 컴포넌트 반환
  call(args: string, context: ToolUseContext): React.ReactNode
}

// 세 타입의 유니온
type Command = PromptCommand | LocalCommand | LocalJSXCommand
```

**타입으로 구분, 경로로 분기:**
- `type: 'prompt'` → 에이전트 루프에 프롬프트를 주입
- `type: 'local'` → 즉시 텍스트 반환
- `type: 'local-jsx'` → 즉시 UI 렌더링

### 5-3. PromptCommand 예시 — /commit (commands/commit.js)

```typescript
// src/commands/commit.js — PromptCommand의 대표적 예시

export const commitCommand: PromptCommand = {
  type: 'prompt',
  name: 'commit',
  description: 'Commit changes with an AI-generated message',
  getPromptForCommand(args: string): string {
    // /commit fix bug → 아래 프롬프트가 LLM에 전달됨
    const userHint = args ? `\nUser hint: ${args}` : ''
    return `Review the current git diff and create a commit.

Instructions:
1. Run \`git diff --staged\` to see staged changes
2. If no staged changes, run \`git diff\` for unstaged changes  
3. Analyze the changes and write a clear commit message
4. Use conventional commit format
5. Run \`git commit -m "<message>"\`
${userHint}`
  }
}
```

**주목할 점:** `/commit`의 구현은 **프롬프트 텍스트**입니다. git 명령을 직접 실행하는 코드가 아니라, LLM에게 "git diff를 보고 커밋하라"고 지시합니다. LLM이 에이전트 루프에서 Bash 도구로 실행합니다.

### 5-4. 커맨드 레지스트리와 조건부 등록 (commands.ts:25+)

```typescript
// src/commands.ts:25+ — 커맨드 레지스트리

// 50개 이상의 커맨드를 배열로 관리
const commands: Command[] = [
  commitCommand,
  reviewCommand,
  costCommand,
  versionCommand,
  helpCommand,
  // ...

  // Feature Flag로 조건부 등록
  // bun:bundle 빌드 시 PROACTIVE === false이면 dead code elimination
  ...(PROACTIVE ? [proactiveCommand] : []),
  ...(KAIROS ? [kairosCommand] : []),
]

// 별칭(alias) 지원: /c → /commit, /h → /help
const aliases: Record<string, string> = {
  'c': 'commit',
  'r': 'review',
  'h': 'help',
  '?': 'help',
}
```

---

## Python으로 구현하기

`mini_claude/commands.py`에 커맨드 시스템을 구현합니다.

### 핵심 구성 요소

1. **Command** ABC — 모든 커맨드의 기본 인터페이스
2. **PromptCommand** — LLM에 보낼 프롬프트를 생성하는 커맨드
3. **LocalCommand** — 로컬에서 즉시 실행되는 커맨드
4. **CommandRegistry** — 등록, 파싱, 별칭, 도움말 생성

In [ ]:
import sys
sys.path.insert(0, ".")

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
import time


# --- 1. Command ABC: 모든 커맨드의 기본 인터페이스 ---

class Command(ABC):
    """커맨드의 기본 인터페이스 — Claude Code의 Command 유니온 타입에 대응"""

    @property
    @abstractmethod
    def name(self) -> str:
        """커맨드 이름 (예: 'commit', 'cost')"""
        ...

    @property
    @abstractmethod
    def description(self) -> str:
        """커맨드 설명 (도움말에 표시)"""
        ...

    @property
    def aliases(self) -> list[str]:
        """별칭 목록 (예: ['c'] for commit)"""
        return []

    @property
    @abstractmethod
    def command_type(self) -> str:
        """'prompt' 또는 'local'"""
        ...


class PromptCommand(Command):
    """LLM에 보낼 프롬프트를 생성하는 커맨드 — /commit, /review 등"""

    @property
    def command_type(self) -> str:
        return "prompt"

    @abstractmethod
    def get_prompt(self, args: str) -> str:
        """사용자 인자를 받아 LLM에 보낼 프롬프트를 반환
        
        Claude Code의 getPromptForCommand(args) 에 대응
        """
        ...


class LocalCommand(Command):
    """LLM 없이 로컬에서 실행되는 커맨드 — /cost, /version 등"""

    @property
    def command_type(self) -> str:
        return "local"

    @abstractmethod
    def execute(self, args: str) -> str:
        """로컬에서 실행하고 텍스트를 반환
        
        Claude Code의 call(args, context) 에 대응
        """
        ...


print("Command ABC 계층 구조:")
print(f"  Command (ABC)")
print(f"    ├─ PromptCommand  -> get_prompt(args) -> str  (LLM에 전달)")
print(f"    └─ LocalCommand   -> execute(args) -> str    (즉시 실행)")
print()
print(f"PromptCommand는 Command의 서브클래스? {issubclass(PromptCommand, Command)}")
print(f"LocalCommand는 Command의 서브클래스?  {issubclass(LocalCommand, Command)}")

### 구체적 커맨드 구현: /commit, /review, /cost, /version

In [ ]:
# --- 2. PromptCommand 구현: /commit ---

class CommitCommand(PromptCommand):
    """git diff를 분석하고 커밋 메시지를 생성하는 커맨드
    
    Claude Code의 src/commands/commit.js 에 대응합니다.
    코드를 직접 실행하지 않고, LLM에게 프롬프트를 전달합니다.
    """

    @property
    def name(self) -> str:
        return "commit"

    @property
    def description(self) -> str:
        return "Commit changes with an AI-generated message"

    @property
    def aliases(self) -> list[str]:
        return ["c"]

    def get_prompt(self, args: str) -> str:
        user_hint = f"\nUser hint: {args}" if args else ""
        return f"""Review the current git diff and create a commit.

Instructions:
1. Run `git diff --staged` to see staged changes
2. If no staged changes, run `git diff` for unstaged changes
3. Analyze the changes and write a clear commit message
4. Use conventional commit format (feat:, fix:, refactor:, etc.)
5. Run `git commit -m "<message>"`
{user_hint}"""


# --- 3. PromptCommand 구현: /review ---

class ReviewCommand(PromptCommand):
    """현재 변경사항을 코드 리뷰하는 커맨드"""

    @property
    def name(self) -> str:
        return "review"

    @property
    def description(self) -> str:
        return "Review current changes or a specific file"

    @property
    def aliases(self) -> list[str]:
        return ["r"]

    def get_prompt(self, args: str) -> str:
        target = f"the file `{args}`" if args else "all current changes (`git diff`)"
        return f"""Review {target} and provide feedback.

Instructions:
1. Read the changes carefully
2. Check for bugs, security issues, and code smells
3. Suggest improvements with specific code examples
4. Be constructive and concise"""


# --- 테스트: PromptCommand의 프롬프트 생성 ---
commit_cmd = CommitCommand()
review_cmd = ReviewCommand()

print("=== /commit fix login bug ===")
print(commit_cmd.get_prompt("fix login bug"))
print()
print("=== /review ===")
print(review_cmd.get_prompt(""))
print()
print(f"커맨드 타입: {commit_cmd.command_type}")
print(f"별칭: {commit_cmd.aliases}")

In [ ]:
# --- 4. LocalCommand 구현: /cost ---

@dataclass
class TokenTracker:
    """토큰 사용량을 추적하는 간단한 트래커"""
    total_input_tokens: int = 0
    total_output_tokens: int = 0
    total_requests: int = 0
    session_start: float = field(default_factory=time.time)

    # 가격 (Claude 3.5 Sonnet 기준, per 1M tokens)
    INPUT_PRICE = 3.0   # $3 / 1M input tokens
    OUTPUT_PRICE = 15.0  # $15 / 1M output tokens

    def add(self, input_tokens: int, output_tokens: int):
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.total_requests += 1

    @property
    def total_cost(self) -> float:
        input_cost = (self.total_input_tokens / 1_000_000) * self.INPUT_PRICE
        output_cost = (self.total_output_tokens / 1_000_000) * self.OUTPUT_PRICE
        return input_cost + output_cost

    @property
    def session_duration(self) -> float:
        return time.time() - self.session_start


# 글로벌 트래커 (세션 단위)
token_tracker = TokenTracker()


class CostCommand(LocalCommand):
    """현재 세션의 토큰 사용량과 비용을 표시하는 커맨드"""

    def __init__(self, tracker: TokenTracker):
        self._tracker = tracker

    @property
    def name(self) -> str:
        return "cost"

    @property
    def description(self) -> str:
        return "Show token usage and estimated cost for this session"

    def execute(self, args: str) -> str:
        t = self._tracker
        duration = t.session_duration
        minutes = int(duration // 60)
        seconds = int(duration % 60)
        return (
            f"Session Cost Summary\n"
            f"{'=' * 40}\n"
            f"  Input tokens:  {t.total_input_tokens:>10,}\n"
            f"  Output tokens: {t.total_output_tokens:>10,}\n"
            f"  Total tokens:  {t.total_input_tokens + t.total_output_tokens:>10,}\n"
            f"  Requests:      {t.total_requests:>10}\n"
            f"  Duration:      {minutes:>7}m {seconds:02d}s\n"
            f"  {'=' * 38}\n"
            f"  Estimated cost: ${t.total_cost:.4f}\n"
        )


# --- 5. LocalCommand 구현: /version ---

class VersionCommand(LocalCommand):
    """버전 정보를 표시하는 커맨드"""

    VERSION = "0.1.0"
    BUILD = "workshop-dev"

    @property
    def name(self) -> str:
        return "version"

    @property
    def description(self) -> str:
        return "Show mini-claude version information"

    @property
    def aliases(self) -> list[str]:
        return ["v"]

    def execute(self, args: str) -> str:
        return f"mini-claude {self.VERSION} ({self.BUILD})"


# --- 테스트: LocalCommand 실행 ---
# 토큰 사용 시뮬레이션
token_tracker.add(input_tokens=1500, output_tokens=350)
token_tracker.add(input_tokens=2200, output_tokens=520)
token_tracker.add(input_tokens=800, output_tokens=150)

cost_cmd = CostCommand(token_tracker)
version_cmd = VersionCommand()

print("=== /cost ===")
print(cost_cmd.execute(""))
print()
print("=== /version ===")
print(version_cmd.execute(""))
print()
print(f"커맨드 타입: {cost_cmd.command_type} (LLM 호출 없이 즉시 실행)")

### CommandRegistry — 커맨드 등록, 파싱, 별칭 관리

In [ ]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class ParsedCommand:
    """파싱된 커맨드 — 커맨드 객체와 인자를 함께 전달"""
    command: Command
    args: str


class CommandRegistry:
    """커맨드 레지스트리 — 등록, 파싱, 별칭, 도움말 생성
    
    Claude Code의 commands.ts 레지스트리에 대응합니다.
    """

    def __init__(self):
        self._commands: dict[str, Command] = {}   # name -> Command
        self._aliases: dict[str, str] = {}         # alias -> name

    def register(self, command: Command) -> None:
        """커맨드를 등록합니다. 별칭도 자동으로 등록됩니다."""
        self._commands[command.name] = command
        for alias in command.aliases:
            self._aliases[alias] = command.name

    def parse(self, user_input: str) -> Optional[ParsedCommand]:
        """사용자 입력을 파싱하여 커맨드와 인자를 분리합니다.
        
        '/commit fix bug' -> ParsedCommand(command=CommitCommand, args='fix bug')
        'hello world'     -> None (커맨드가 아님)
        '/unknown'        -> None (등록되지 않은 커맨드)
        """
        stripped = user_input.strip()
        if not stripped.startswith("/"):
            return None

        # '/commit fix bug' -> cmd_name='commit', args='fix bug'
        parts = stripped[1:].split(" ", 1)
        cmd_name = parts[0].lower()
        args = parts[1] if len(parts) > 1 else ""

        # 별칭 해석: /c -> /commit
        resolved_name = self._aliases.get(cmd_name, cmd_name)

        command = self._commands.get(resolved_name)
        if command is None:
            return None

        return ParsedCommand(command=command, args=args)

    def get_help_text(self) -> str:
        """등록된 모든 커맨드의 도움말을 생성합니다."""
        lines = ["Available Commands:", "=" * 50]
        for cmd in sorted(self._commands.values(), key=lambda c: c.name):
            alias_str = ""
            if cmd.aliases:
                alias_str = f" (aliases: {', '.join('/' + a for a in cmd.aliases)})"
            type_badge = "[prompt]" if cmd.command_type == "prompt" else "[local] "
            lines.append(f"  /{cmd.name:<12} {type_badge} {cmd.description}{alias_str}")
        return "\n".join(lines)

    def get_all_commands(self) -> list[Command]:
        """등록된 모든 커맨드를 반환합니다."""
        return list(self._commands.values())

    def __len__(self) -> int:
        return len(self._commands)


# --- 레지스트리 생성 및 커맨드 등록 ---
registry = CommandRegistry()
registry.register(CommitCommand())
registry.register(ReviewCommand())
registry.register(CostCommand(token_tracker))
registry.register(VersionCommand())

print(f"등록된 커맨드: {len(registry)}개")
print()
print(registry.get_help_text())

### Feature Flag 조건부 등록

Claude Code는 `bun:bundle`의 Feature Flag로 빌드 시점에 dead code elimination을 수행합니다.
Python에서는 환경변수나 설정으로 동일한 패턴을 구현할 수 있습니다.

In [ ]:
import os

# --- Feature Flag 패턴 ---
# Claude Code: ...(PROACTIVE ? [proactiveCommand] : [])
# Python: 환경변수로 조건부 등록

# 시뮬레이션용 Feature Flags
FEATURE_FLAGS = {
    "EXPERIMENTAL_COMMANDS": True,
    "PROACTIVE": False,      # 빌드에서 비활성화된 기능
    "DEBUG_MODE": True,
}


class DebugCommand(LocalCommand):
    """디버그 정보를 표시하는 커맨드 (DEBUG_MODE일 때만 활성화)"""

    @property
    def name(self) -> str:
        return "debug"

    @property
    def description(self) -> str:
        return "Show debug information (debug mode only)"

    def execute(self, args: str) -> str:
        return (
            f"Debug Info\n"
            f"  Feature Flags: {FEATURE_FLAGS}\n"
            f"  Registered commands: {len(registry)}\n"
            f"  Python: {sys.version.split()[0]}\n"
        )


class ExplainCommand(PromptCommand):
    """코드를 설명하는 커맨드 (EXPERIMENTAL일 때만 활성화)"""

    @property
    def name(self) -> str:
        return "explain"

    @property
    def description(self) -> str:
        return "Explain a piece of code or concept"

    @property
    def aliases(self) -> list[str]:
        return ["e"]

    def get_prompt(self, args: str) -> str:
        return f"""Explain the following in simple terms:

{args}

Provide:
1. A brief summary (1-2 sentences)
2. Key concepts involved
3. A simple analogy if possible"""


# --- 조건부 등록 (Claude Code의 spread 패턴과 동일) ---
conditional_commands: list[tuple[str, Command]] = [
    ("DEBUG_MODE", DebugCommand()),
    ("EXPERIMENTAL_COMMANDS", ExplainCommand()),
]

for flag_name, cmd in conditional_commands:
    if FEATURE_FLAGS.get(flag_name, False):
        registry.register(cmd)
        print(f"  [ON]  /{cmd.name} 등록됨 ({flag_name}=True)")
    else:
        print(f"  [OFF] /{cmd.name} 건너뜀 ({flag_name}=False)")

print()
print(f"최종 등록된 커맨드: {len(registry)}개")
print()
print(registry.get_help_text())

---

## 실습: 커맨드 파싱과 실행

다양한 입력을 파싱하여 커맨드 시스템이 올바르게 동작하는지 확인합니다.

In [ ]:
# --- 파싱 테스트 ---
test_inputs = [
    "/commit fix bug",        # PromptCommand + 인자
    "/c quick fix",            # 별칭 /c -> /commit
    "/review",                 # PromptCommand, 인자 없음
    "/cost",                   # LocalCommand
    "/version",                # LocalCommand
    "/v",                      # 별칭 /v -> /version
    "regular message",         # 커맨드가 아님
    "/unknown do something",   # 등록되지 않은 커맨드
    "/debug",                  # Feature Flag로 등록된 커맨드
    "/explain recursion",      # Feature Flag로 등록된 커맨드
]

print("커맨드 파싱 테스트")
print("=" * 70)
for user_input in test_inputs:
    parsed = registry.parse(user_input)
    if parsed is None:
        if user_input.startswith("/"):
            status = "[NOT FOUND]"
        else:
            status = "[NOT A CMD]"
        print(f"  {user_input:<30} -> {status}")
    else:
        cmd = parsed.command
        args_display = f'args="{parsed.args}"' if parsed.args else "(no args)"
        print(
            f"  {user_input:<30} -> /{cmd.name} [{cmd.command_type}] {args_display}"
        )

### 실습: 에이전트 루프와 커맨드 시스템 통합

`process_input()` 함수가 사용자 입력을 받아 커맨드인지 일반 메시지인지 판단하고, 적절한 경로로 분기합니다. 이것이 Claude Code의 입력 처리 파이프라인입니다.

In [ ]:
from dataclasses import dataclass


@dataclass
class ProcessResult:
    """입력 처리 결과"""
    type: str          # 'prompt_to_llm', 'local_result', 'regular_message'
    content: str       # 프롬프트, 로컬 실행 결과, 또는 원본 메시지
    command_name: str = ""  # 실행된 커맨드 이름 (있는 경우)


def process_input(user_input: str, cmd_registry: CommandRegistry) -> ProcessResult:
    """사용자 입력을 처리하여 적절한 경로로 분기합니다.
    
    Claude Code의 입력 처리 파이프라인:
    1. 커맨드인지 확인 (/ 로 시작하는지)
    2. PromptCommand면 -> 프롬프트를 생성하여 LLM에 전달
    3. LocalCommand면  -> 즉시 실행하여 결과 반환
    4. 커맨드가 아니면  -> 일반 메시지로 LLM에 전달
    """
    parsed = cmd_registry.parse(user_input)

    if parsed is None:
        # 커맨드가 아님 -> 일반 메시지로 LLM에 전달
        return ProcessResult(
            type="regular_message",
            content=user_input,
        )

    cmd = parsed.command

    if isinstance(cmd, PromptCommand):
        # PromptCommand -> 프롬프트를 생성하여 에이전트 루프에 전달
        prompt = cmd.get_prompt(parsed.args)
        return ProcessResult(
            type="prompt_to_llm",
            content=prompt,
            command_name=cmd.name,
        )

    elif isinstance(cmd, LocalCommand):
        # LocalCommand -> 즉시 실행하여 결과 반환
        result = cmd.execute(parsed.args)
        return ProcessResult(
            type="local_result",
            content=result,
            command_name=cmd.name,
        )

    # 여기에 도달하면 안 됨
    return ProcessResult(type="unknown", content="")


# --- 통합 테스트: REPL 시뮬레이션 ---
print("Mini Claude REPL 시뮬레이션")
print("=" * 60)

simulated_inputs = [
    "/commit fix login bug",
    "/cost",
    "/version",
    "파일 목록을 보여줘",
    "/review main.py",
]

for user_input in simulated_inputs:
    result = process_input(user_input, registry)
    print(f"\n입력: {user_input}")
    print(f"  타입: {result.type}")
    if result.command_name:
        print(f"  커맨드: /{result.command_name}")

    if result.type == "prompt_to_llm":
        # PromptCommand: 프롬프트를 LLM에 전달
        print(f"  -> LLM에 전달될 프롬프트 (첫 80자):")
        print(f"     '{result.content[:80]}...'")
    elif result.type == "local_result":
        # LocalCommand: 즉시 결과 표시
        first_line = result.content.split("\n")[0]
        print(f"  -> 즉시 결과: {first_line}")
    elif result.type == "regular_message":
        # 일반 메시지: 그대로 LLM에 전달
        print(f"  -> 일반 메시지로 LLM에 전달: '{result.content}'")

In [ ]:
# --- PromptCommand가 에이전트 루프에서 어떻게 처리되는지 상세 시뮬레이션 ---

print("PromptCommand의 에이전트 루프 통합 흐름")
print("=" * 60)

# 1. 사용자 입력
user_input = "/commit fix login bug"
print(f"\n[1] 사용자 입력: {user_input}")

# 2. 커맨드 파싱
parsed = registry.parse(user_input)
print(f"[2] 파싱 결과: /{parsed.command.name} (type={parsed.command.command_type})")
print(f"    인자: '{parsed.args}'")

# 3. 프롬프트 생성
prompt = parsed.command.get_prompt(parsed.args)
print(f"[3] 생성된 프롬프트:")
for line in prompt.strip().split("\n"):
    print(f"    | {line}")

# 4. 에이전트 루프에 전달 (시뮬레이션)
print(f"\n[4] 에이전트 루프에 전달:")
message = {"role": "user", "content": prompt}
print(f"    messages.append({{'role': 'user', 'content': '<prompt>'}})")
print(f"    -> LLM API 호출")
print(f"    -> LLM이 Bash 도구로 git diff, git commit 실행")
print(f"    -> 결과를 사용자에게 표시")

print(f"\n핵심: /commit의 '구현'은 프롬프트 텍스트입니다.")
print(f"      git 관련 코드는 한 줄도 없고, LLM이 도구를 사용하여 작업합니다.")

---

## 핵심 설계 원칙 정리

### 1. 프롬프트가 코드를 대체
PromptCommand는 잘 작성된 프롬프트 하나로 복잡한 워크플로우를 구현합니다. `/commit`은 git 명령을 직접 실행하는 코드가 아니라, LLM에게 "git diff를 분석하고 커밋하라"는 지시입니다. 프롬프트 엔지니어링이 곧 기능 구현입니다.

### 2. 타입으로 구분, 경로로 분기
3가지 타입(PromptCommand, LocalCommand, LocalJSXCommand)이 각각 다른 실행 경로를 탑니다. PromptCommand는 에이전트 루프로, LocalCommand는 즉시 실행으로, LocalJSXCommand는 UI 렌더링으로. `command_type` 필드 하나로 분기가 결정됩니다.

### 3. 별칭으로 접근성
`/c` → `/commit`, `/h` → `/help`처럼 짧은 별칭을 지원합니다. 자주 사용하는 커맨드를 빠르게 호출할 수 있어 개발자 경험(DX)을 높입니다. 별칭은 커맨드 객체에 선언되어 레지스트리가 자동으로 등록합니다.

### 4. Feature Flag로 조건부 기능
Claude Code는 `bun:bundle`의 Feature Flag로 빌드 시점에 dead code elimination을 수행합니다. `PROACTIVE`, `KAIROS` 같은 플래그가 `false`이면 해당 커맨드는 빌드에서 완전히 제거됩니다. Python에서는 환경변수나 설정 파일로 동일한 패턴을 구현합니다.

---

## 다음 Step 미리보기

**Step 6: 컨텍스트 관리**에서는 시스템 프롬프트와 컨텍스트를 효율적으로 관리하는 방법을 배웁니다:
- **정적/동적 경계**로 프롬프트 캐시를 최적화하는 방법
- `@lru_cache`(memoize)로 반복 계산을 피하는 방법
- `ToolUseContext`가 도구 실행 환경을 캡슐화하는 패턴

커맨드 시스템으로 사용자 인터페이스가 완성되었습니다. Step 6에서는 시스템 프롬프트와 컨텍스트를 효율적으로 관리하는 방법을 배웁니다.